In [0]:
import mlflow
import mlflow.spark

from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

df = spark.table(
    "proyecto_final_mlops.telco_database.churn_dataset_training_data"
)

print("Registros:", df.count())
display(df.limit(5))

Registros: 440832


CustomerID,Age,Gender,Tenure,Usage_Frequency,Support_Calls,Payment_Delay,Subscription_Type,Contract_Length,Total_Spend,Last_Interaction,Churn
2,30,Female,39,14,5,18,Standard,Annual,932.0,17,1
3,65,Female,49,1,10,8,Basic,Monthly,557.0,6,1
4,55,Female,14,4,6,18,Basic,Quarterly,185.0,3,1
5,58,Male,38,21,7,7,Standard,Monthly,396.0,29,1
6,23,Male,32,20,5,8,Basic,Monthly,617.0,20,1


In [0]:
train_df, validation_df = df.randomSplit([0.8, 0.2],seed=42)

print("Train:", train_df.count())
print("Validation:", validation_df.count())

Train: 352915
Validation: 87917


In [0]:
categorical_cols = [
    "Gender",
    "Subscription_Type",
    "Contract_Length"
]

numeric_cols = [
    "Age",
    "Tenure",
    "Usage_Frequency",
    "Support_Calls",
    "Payment_Delay",
    "Total_Spend",
    "Last_Interaction"
]

In [0]:
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_encoded" for c in categorical_cols]
)

assembler = VectorAssembler(
    inputCols=
        numeric_cols +
        [f"{c}_encoded" for c in categorical_cols],
    outputCol="features"
)

In [0]:
logit = LogisticRegression(
    featuresCol="features",
    labelCol="Churn",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline = Pipeline(
    stages=indexers + [encoder, assembler, logit]
)

In [0]:
usuario = spark.sql("SELECT current_user()").first()[0]

experiment_name = f"/Users/{usuario}/customer_churn_logit"

mlflow.set_experiment(experiment_name)

print("Experiment:", experiment_name)

Experiment: /Users/michaelarmasb@gmail.com/customer_churn_logit


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

with mlflow.start_run(run_name="logit_baseline") as run:

    # Entrenar
    model = pipeline.fit(train_df)

    # Validación
    predictions = model.transform(validation_df)

    # ROC AUC
    roc_evaluator = BinaryClassificationEvaluator(
        labelCol="Churn",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    roc_auc = roc_evaluator.evaluate(predictions)

    # PR AUC
    pr_evaluator = BinaryClassificationEvaluator(
        labelCol="Churn",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )

    pr_auc = pr_evaluator.evaluate(predictions)

    # Matriz de confusión
    tp = predictions.filter(
        (col("Churn") == 1) & (col("prediction") == 1)
    ).count()

    tn = predictions.filter(
        (col("Churn") == 0) & (col("prediction") == 0)
    ).count()

    fp = predictions.filter(
        (col("Churn") == 0) & (col("prediction") == 1)
    ).count()

    fn = predictions.filter(
        (col("Churn") == 1) & (col("prediction") == 0)
    ).count()

    # Métricas
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    # Parámetros MLflow
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("train_split", 0.80)
    mlflow.log_param("validation_split", 0.20)
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("regParam", 0.0)
    mlflow.log_param("elasticNetParam", 0.0)

    # Métricas MLflow
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    # Guardar modelo dentro del Run
    mlflow.spark.log_model(
        model,
        artifact_path="model",
        dfs_tmpdir="/Volumes/proyecto_final_mlops/telco_database/tmp"
    )

    run_id = run.info.run_id

print("Run ID:", run_id)

2026/09/07 01:06:17 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.9) contains a local version label (+databricks.connect.18.0.9). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/09/07 01:06:19 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-9090af48-d071-499c-beae-30/tmpx4wtrjjr/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/09/07 01:06:19 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Run ID: db1c5520df114070bc88b4c17cc6f38e


In [0]:
print(f"ROC AUC   : {roc_auc:.4f}")
print(f"PR AUC    : {pr_auc:.4f}")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")

ROC AUC   : 0.9608
PR AUC    : 0.9751
Accuracy  : 0.8978
Precision : 0.9267
Recall    : 0.8901
F1        : 0.9080


In [0]:
confusion_matrix = (
    predictions
    .groupBy("Churn", "prediction")
    .count()
    .orderBy("Churn", "prediction")
)

display(confusion_matrix)

Churn,prediction,count
0,0.0,34594
0,1.0,3506
1,0.0,5475
1,1.0,44342


In [0]:
from mlflow.models.signature import infer_signature

mlflow.set_registry_uri("databricks-uc")

model_name = (
    "proyecto_final_mlops."
    "telco_database."
    "churn_logit_model"
)

# Infer signature from predictions
input_example = validation_df.limit(1).toPandas()
sample_input = validation_df.select(categorical_cols + numeric_cols).limit(1).toPandas()
sample_output = predictions.select("prediction", "probability").limit(1).toPandas()
signature = infer_signature(sample_input, sample_output)

model_uri = f"runs:/{run_id}/model"

# Re-log the model with signature
with mlflow.start_run(run_id=run_id):
    mlflow.spark.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=sample_input,
        dfs_tmpdir="/Volumes/proyecto_final_mlops/telco_database/tmp"
    )

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print("Modelo:", model_name)
print("Versión:", registered_model.version)

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/07 01:08:15 WARNING mlflow.models.signature: Failed to infer schema for outputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. To see the full traceback, set logging level to DEBUG.
2026/09/07 01:08:35 WARNING mlf

Uploading artifacts:   0%|          | 0/94 [00:00<?, ?it/s]

Modelo: proyecto_final_mlops.telco_database.churn_logit_model
Versión: 1


🔗 Created version '1' of model 'proyecto_final_mlops.telco_database.churn_logit_model': https://dbc-ebb9ad9e-bbdf.cloud.databricks.com/explore/data/models/proyecto_final_mlops/telco_database/churn_logit_model/version/1?o=7474653079374067


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

def calcular_metricas(model, df, nombre):

    pred = model.transform(df)

    # AUC
    roc_auc = BinaryClassificationEvaluator(
        labelCol="Churn",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    ).evaluate(pred)

    pr_auc = BinaryClassificationEvaluator(
        labelCol="Churn",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    ).evaluate(pred)

    # Matriz de confusión
    tp = pred.filter((col("Churn") == 1) & (col("prediction") == 1)).count()
    tn = pred.filter((col("Churn") == 0) & (col("prediction") == 0)).count()
    fp = pred.filter((col("Churn") == 0) & (col("prediction") == 1)).count()
    fn = pred.filter((col("Churn") == 1) & (col("prediction") == 0)).count()

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return (
        nombre,
        roc_auc,
        pr_auc,
        accuracy,
        precision,
        recall,
        f1
    )


metricas_train = calcular_metricas(model, train_df, "Train 80%")
metricas_validation = calcular_metricas(model, validation_df, "Validation 20%")

resultados = spark.createDataFrame(
    [metricas_train, metricas_validation],
    [
        "Dataset",
        "ROC_AUC",
        "PR_AUC",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

display(resultados)

Dataset,ROC_AUC,PR_AUC,Accuracy,Precision,Recall,F1
Train 80%,0.959275197301006,0.9741927275038651,0.8948245328195175,0.924707769883108,0.886783027445025,0.9053484090433861
Validation 20%,0.9607904168100395,0.9751134701211625,0.8978468328082169,0.9267262999498411,0.8900977577935243,0.9080427993651768
